# S(Ap, x): Gráfico 2D por Segmentos y Superficie 3D

Este notebook combina **dos enfoques** para visualizar la función $S(A_p, x)$:

| Bloque | Descripción |
|:------:|:------------|
| **1 – Gráfico 2D por segmentos** | Divide el dominio en tres intervalos para evitar las singularidades en $x = 0.25$ y $x \approx 0.711$ (igual que *Untitled.ipynb*). |
| **2 – Superficie 3D** | Evalúa $S$ sobre una malla $(A_p, x)$ y la visualiza con `plot_surface` y `contourf` (igual que *Codigo2.ipynb*). |

## Contexto físico

La función $S(A_p, x)$ modela la distribución de señal de partículas en función
de la fracción de momento $x$ y el parámetro de amplitud $A_p$.  Este notebook
permite explorar simultáneamente la dependencia en $x$ (Bloque 1) y en el espacio
bidimensional $(A_p, x)$ (Bloque 2).

## Parámetros

| Variable | Valor | Descripción |
|:--------:|:-----:|:------------|
| `Ap`     | 0.8   | Parámetro de amplitud $A_p$ (fijo en el Bloque 1) |
| `d`      | 10    | Parámetro de estructura $d$ |
| `n`      | 0.5   | Índice $n$ de la distribución |
| `B`      | 0.1   | Parámetro $\beta$ — régimen $\beta \ll 1$ |

## Singularidades del dominio

| Singularidad | Causa |
|:---:|:---|
| $x = 0.25$ | $1-4x = 0$ → divergencia en el numerador |
| $x \approx 0.711$ | El denominador de $S$ se anula para los parámetros dados |
| $x = 1$ | Límite físico ($1-x = 0$) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 1: Gráfico 2D por segmentos de S(x)   (Ap fijo = 0.8)
# ══════════════════════════════════════════════════════════════════════════════

# ── Parámetros del modelo ─────────────────────────────────────────────────────
Ap = 0.8   # Parámetro de amplitud A_p (constante en este bloque)
d  = 10    # Parámetro de estructura d
n  = 0.5   # Índice n de la distribución
B  = 0.1   # β (régimen β << 1)

# ── División del dominio en tres segmentos (evitando singularidades) ──────────
# Segmento 1: antes de la singularidad estructural x = 0.25
xa = np.linspace(0.1,   0.250, 300)
# Segmento 2: entre las singularidades x = 0.25 y x ≈ 0.711
xb = np.linspace(0.251, 0.711, 300)
# Segmento 3: después de la singularidad paramétrica
xc = np.linspace(0.712, 0.99,  300)

# ── Fórmula de S(x) simplificada (cociente polinomial con corrección β) ────────
# S(x) = [ (1+d)/(1-x) + 4n/(1-4x) ] / [ 1 + β/(2(1-x)²)·(Ap - x/(1-x)) ]
def S_2d(x):
    """Forma simplificada de S(x) para Ap, d, n, B globales."""
    num = (1 + d) / (1 - x) + (4 * n) / (1 - 4 * x)
    den = 1 + (B / (2 * (1 - x) ** 2)) * (Ap - x / (1 - x))
    return num / den

Sa = S_2d(xa)
Sb = S_2d(xb)
Sc = S_2d(xc)

# ── Posiciones de las singularidades ──────────────────────────────────────────
x_sing1, x_sing2 = 0.25, 0.711

# ── Gráfico 2D ────────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.plot(xa, Sa, label='Segmento 1  (0.100 – 0.250)')
plt.plot(xb, Sb, label='Segmento 2  (0.251 – 0.711)')
plt.plot(xc, Sc, label='Segmento 3  (0.712 – 0.990)')
plt.axvline(x_sing1, color='blue',   linestyle='--', label=f'x = {x_sing1}  (singularidad)')
plt.axvline(x_sing2, color='blue',   linestyle='--', label=f'x = {x_sing2}  (singularidad)')
plt.axvline(0,       color='orange', linestyle='--', label='x = 0')
plt.axvline(1,       color='orange', linestyle='--', label='x = 1')
plt.xlabel('x', fontsize=12)
plt.ylabel('S(x)', fontsize=12)
plt.title('S(x) por segmentos  —  Ap = 0.8, régimen β << 1', fontsize=13)
plt.legend(fontsize=9)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 2: Superficie 3D de S(Ap, x)
# ══════════════════════════════════════════════════════════════════════════════


def S(Ap, x):
    """
    Calcula la función completa S(Ap, x) en el régimen β << 1.

    Fórmula:
        S = (√π / 2a) · Ω · [Ap(1-x) - x]
            · exp( -(1-x)²/β² · NUM_EXP / DEN_EXP )
            · [x² · DEN_EXP]⁻¹

    donde:
        NUM_EXP = 1 + (β/2) · [Ap(1-x) - x] / (1-x)³
        DEN_EXP = (1+d)/(1-x) + 4n/(1-4x)

    Parámetros
    ----------
    Ap : float o ndarray
        Parámetro de amplitud A_p.
    x : float o ndarray
        Variable cinemática. Evitar x = 0.25 y x = 1.

    Devuelve
    --------
    float o ndarray
        Valor de S en el/los puntos indicados.
    """
    # ── Parámetros internos ────────────────────────────────────────────────
    a = 1    # Constante de normalización
    O = 1    # Factor Ω (Omega)
    d = 10   # Parámetro de estructura d
    n = 0.5  # Índice n de la distribución
    B = 0.1  # β (régimen β << 1)

    # ── Términos auxiliares ───────────────────────────────────────────────
    amp      = Ap * (1 - x) - x
    den_poly = (1 + d) / (1 - x) + (4 * n) / (1 - 4 * x)
    num_exp  = 1 + (B / 2) * amp / (1 - x) ** 3
    exponent = -((1 - x) ** 2 / B ** 2) * (num_exp / den_poly)

    prefactor = (np.sqrt(np.pi) / (2 * a)) * O
    return prefactor * amp * np.exp(exponent) * (x ** 2 * den_poly) ** (-1)


# ── Malla (Ap, x) ─────────────────────────────────────────────────────────────
# x evita x = 0.25 usando dos sub-rangos concatenados
Ap_vec = np.linspace(0.1, 3.0, 25)
x_vec  = np.concatenate([np.linspace(0.1, 0.24, 12), np.linspace(0.26, 0.99, 13)])
Ap_grid, x_grid = np.meshgrid(Ap_vec, x_vec)   # Mallas 2D
S_grid = S(Ap_grid, x_grid)                     # Evaluación vectorizada

# ── Superficie 3D ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(12, 8))
ax  = fig.add_subplot(111, projection='3d')
ax.plot_surface(Ap_grid, x_grid, S_grid, cmap='cool', alpha=0.85)
# Proyecciones de contorno en las paredes del gráfico
ax.contourf(Ap_grid, x_grid, S_grid, zdir='x', offset=Ap_vec.min() - 0.5, cmap='coolwarm')
ax.contourf(Ap_grid, x_grid, S_grid, zdir='y', offset=x_vec.max()  + 0.05, cmap='coolwarm')
ax.set_xlabel('Ap', fontsize=11)
ax.set_ylabel('x',  fontsize=11)
ax.set_zlabel('S',  fontsize=11)
ax.set_title('Superficie 3D de S(Ap, x)  —  régimen β << 1', fontsize=13)
plt.tight_layout()
plt.show()
